# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_07 — Generación de respuestas con `ai.generate_response`

**Serie:** Microsoft Fabric AI Functions con Pandas.

Objetivos: generar borradores de soporte, comparar prompts y formatos estructurados, validar resultados y auditar tokens y CU-segundos. Requiere runtime compatible, permisos y capacidad Fabric. Usar datos ficticios y revisión humana.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 3, Finished, Available, Finished, False)

## 2. Configuración
Modelo gpt-5-mini, concurrencia de laboratorio 10 y barra de progreso con estadísticas. [Configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration).

In [2]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "gpt-5-mini"
CONCURRENCY = 10
aifunc.default_conf.model_deployment_name = MODEL_NAME
aifunc.default_conf.reasoning_effort = "low"
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.generate_response con pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/generate-response)

```python
df.ai.generate_response(prompt="Redacta una respuesta")
df.ai.generate_response(prompt="Responde a {solicitud}", is_prompt_template=True)
```

| Parámetro | Tipo | Uso |
|---|---|---|
| `prompt` | str | Instrucciones obligatorias. |
| `is_prompt_template` | bool | False por defecto: contexto completo; True: campos entre llaves. |
| `response_format` | None, str, dict o BaseModel | Formato de salida; None por defecto. |
| `column_type` | str | Para Series con rutas de archivos. |
| `column_type_dict` | dict | Para DataFrame con columnas de rutas. |

Devuelve una Series por fila, incluso para JSON (texto que debe interpretarse). Formatos: `text`, `json_object` (incluir JSON en el prompt), `json_schema` y clase Pydantic. Los ejercicios siguientes son propios. No enviar secretos ni datos personales.

## 4. Dataset ficticio
Los cuatro tickets originales de PySpark, sin modificar.

In [3]:
data = [(1, "Alta", "El informe de ventas no se actualiza", "En investigación"),
        (2, "Media", "Necesito acceso al tablero", "Pendiente de autorización"),
        (3, "Alta", "Falló la carga nocturna", "Equipo técnico informado"),
        (4, "Baja", "Solicito documentación del catálogo", "Disponible")]
df = pd.DataFrame(data, columns= ["id_ticket", "prioridad", "solicitud", "estado"])
display(df)

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d900bdf8-1659-4c8a-8019-58cc26515f38)

## 5. Prompt simple
Usar todas las columnas originales. Registrar estadísticas inmediatamente y medir tiempo y memoria aproximada.

In [4]:
runs = []
def ejecutar(nombre, entrada, **kwargs):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc).isoformat()
    input_bytes = int(entrada.memory_usage(deep=True).sum() if isinstance(entrada, pd.DataFrame) else entrada.memory_usage(deep=True))
    t0 = perf_counter()
    resultado = entrada.ai.generate_response(**kwargs)
    elapsed = perf_counter() - t0
    raw = resultado.ai.stats
    runs.append((nombre, run_id, started, elapsed, raw, resultado, len(entrada), input_bytes))
    return resultado


respuesta = ejecutar("texto", df, prompt="Redacta un borrador de respuesta de soporte en español, máximo 60 palabras. Usa solo los datos disponibles; no prometas plazos ni acciones no confirmadas.", response_format="text")
display(df.assign(respuesta=respuesta))

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 6, Finished, Available, Finished, False)

ai.generate_response: 100%|██████████| 4/4 [00:03<00:00,  1.02it/s, cached=0, in=418, out=1.42k, CU-s=95.68]                          


SynapseWidget(Synapse.DataFrame, 7325da0f-805d-409b-9f10-a83ff4581fef)

## 6. Plantilla de prompt (opcional)
Solo los campos nombrados se incluyen como contexto. No usar f-string. Cada llamada adicional consume capacidad y se incluye en la auditoría.

In [5]:
RUN_TEMPLATE = False
if RUN_TEMPLATE:
    plantilla = ejecutar("plantilla", df, prompt="Redacta un acuse de recibo para {solicitud}. Estado confirmado: {estado}. No inventes datos.", is_prompt_template=True, response_format="text")
    display(df.assign(respuesta_plantilla=plantilla))

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 7, Finished, Available, Finished, False)

## 7. Salidas estructuradas y validación
Activar individualmente los ejemplos JSON y Pydantic. El esquema garantiza estructura cuando se cumple, no exactitud factual. Validar el texto principal y revisar manualmente cada borrador. Las respuestas no deben enviarse automáticamente.

In [6]:
import json
RUN_JSON = False
RUN_PYDANTIC = False
RUN_JSON_OBJECT = False
formato = {"type": "json_schema", "json_schema": {"name": "respuesta_soporte", "strict": True, "schema": {"type": "object", "properties": {"respuesta": {"type": "string"}, "requiere_revision": {"type": "boolean"}}, "required": ["respuesta", "requiere_revision"], "additionalProperties": False}}}

def validar_json(serie):
    filas = []
    for valor in serie:
        try:
            obj = json.loads(valor)
            valido = isinstance(obj, dict) and set(obj) == {"respuesta", "requiere_revision"} and isinstance(obj["respuesta"], str) and obj["requiere_revision"] is True
            filas.append({"objeto": obj, "valido": valido})
        except (ValueError, TypeError):
            filas.append({"objeto": None, "valido": False})
    return pd.DataFrame(filas, index=serie.index)

prompt_json = "Genera un borrador de soporte basado únicamente en los datos, en JSON, con respuesta de texto y requiere_revision igual a true."
if RUN_JSON:
    estructurado = ejecutar("json_schema", df, prompt=prompt_json, response_format=formato)
    display(df.join(validar_json(estructurado)))
if RUN_JSON_OBJECT:
    objeto = ejecutar("json_object", df, prompt=prompt_json, response_format="json_object")
    display(df.join(validar_json(objeto)))
if RUN_PYDANTIC:
    from pydantic import BaseModel
    class RespuestaSoporte(BaseModel):
        respuesta: str
        requiere_revision: bool
    salida = ejecutar("pydantic", df, prompt=prompt_json, response_format=RespuestaSoporte)
    display(df.join(validar_json(salida)))
validacion = df.assign(respuesta=respuesta)
validacion["palabras"] = respuesta.astype("string").str.split().str.len()
validacion["supera_limite"] = validacion.palabras.gt(60)
validacion["estado_validacion"] = "REVISAR"
validacion.loc[respuesta.isna() | respuesta.astype("string").str.strip().eq("").fillna(False), "estado_validacion"] = "SIN_RESPUESTA"
display(validacion)

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 882a8b62-01c7-4e40-a288-2b099fd7d8fa)

## 8. Estadísticas y tokens
Consultar cada ejecución por separado. No reemplazar métricas ausentes por cero.

In [7]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")
for nombre, _, _, _, raw, _, _, _ in runs:
    print(nombre)
    display(raw)

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 9, Finished, Available, Finished, False)

texto


SynapseWidget(Synapse.DataFrame, c35a4db9-025b-4d80-8058-2de33c48165f)

## 9. Estimación de CU-segundos
Tarifas de referencia de la serie para gpt-5-mini integrado, en CU-s por 1.000 tokens:

| Entrada sin caché | Entrada en caché | Salida |
|---:|---:|---:|
| 8,40 | 0,84 | 67,23 |

$CU_s=((I-C)\times8.40+C\times0.84+O\times67.23)/1000$

C es un subconjunto de I. Los tokens de razonamiento no se suman nuevamente a O. Ejemplo hipotético: I=1000, C=200, O=100 da 13,611 CU-s. No representa una ejecución real ni el consumo total del runtime. Verificar vigencia en [facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing). Si faltan métricas o el modelo/cliente no coincide, no estimar.

## 10. Auditoría individual y acumulada
Una fila por llamada habilitada. No se guardan textos. El estado es técnico, no calidad de respuesta. Las métricas ausentes permanecen nulas.

In [8]:
records = []
for nombre, run_id, started, elapsed, raw, resultado_run, num_rows, input_bytes in runs:
    stats = normalizar(raw)
    def metric(k):
        v = stats.get(k)
        return None if v is None or pd.isna(v) else int(v)
    I,C,O = [metric(k) for k in ("input_tokens","cached_tokens","output_tokens")]
    cu = None
    if (stats.get("model") in (MODEL_NAME,"gpt-5-mini-2025-08-07") and stats.get("client_type")=="fabric_llm_endpoint" and all(v is not None for v in (I,C,O)) and 0<=C<=I and O>=0):
        cu = ((I-C)*8.40+C*0.84+O*67.23)/1000
    audit = dict(run_id=run_id, execution_timestamp_utc=started, notebook_name="NB_07_AI_GENERATE_RESPONSE_PANDAS", operation="ai.generate_response."+nombre, model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=num_rows, client_type=stats.get("client_type"), estimated_cu_seconds=cu, elapsed_seconds=elapsed, input_bytes=input_bytes, output_bytes=int(resultado_run.memory_usage(deep=True)), concurrency=CONCURRENCY)
    for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
        audit[key]=metric(key)
    expected = num_rows
    if (audit["num_exceptions"] or 0) > 0 or (audit["num_harmful"] or 0) > 0:
        audit["execution_status"] = "PARTIAL"
    elif audit["num_successful"] == expected and audit["num_exceptions"] == 0 and audit["num_harmful"] == 0 and resultado_run.notna().all():
        audit["execution_status"] = "SUCCESS"
    else:
        audit["execution_status"] = "INCOMPLETE"
    records.append(audit)
audit_df=pd.DataFrame(records)
display(audit_df)
print("CU-s acumulados:", audit_df.estimated_cu_seconds.sum(min_count=len(audit_df)))

StatementMeta(, 770b2ba1-8d4c-4242-9810-884e2c77c9fa, 10, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 22e8ef2a-68a2-4ae7-93a3-31cd8f0f8ea9)

CU-s acumulados: 98.84334


## 11. Persistencia opcional en Delta
Requiere Lakehouse adjunto, deltalake y pyarrow. Persiste en Files, no registra una tabla SQL. Deshabilitada por defecto.

In [ ]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

## 12. Multimodalidad (opcional)
Usar archivos autorizados y revisar [requisitos multimodales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). Consumo independiente, fuera de la auditoría principal.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    archivos = pd.Series(["/lakehouse/default/Files/ejemplo.pdf"])
    resultado_archivos = archivos.ai.generate_response(prompt="Describe el documento sin inventar datos.", column_type="path")
    display(resultado_archivos)
    display(resultado_archivos.ai.stats)

## 13. Comparación y conclusiones

| Pandas | PySpark |
|---|---|
| `df.ai.generate_response(prompt="...")` | `df.ai.generate_response(prompt="...", output_col="respuesta", error_col="error")` |
| Series de texto o JSON | Spark DataFrame |
| Memoria local | Procesamiento distribuido |

Se conserva el dataset original. Revisar fidelidad, promesas no confirmadas y JSON. La auditoría incluye las llamadas de texto y formatos opcionales habilitados; multimodalidad se muestra por separado. No representa el consumo total de capacidad.

- [Método oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/generate-response)
- [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

Validación estática local; pendiente ejecución en Fabric y comprobación de vigencia de tarifas.